# 04 — Model Comparison

Compare performance across all 8 models: 6 local (Ollama) + GPT-4o + Claude 3.5 Sonnet.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

with open('../results/experiment_results.json') as f:
    data = json.load(f)

## Model Performance Under sc+rag+3r (Best Condition)

In [ ]:
# sc+rag+3r performance for all models
main = data.get('main_results', {})
models_display = {
    'deepseek':  'DeepSeek Coder v2\n(16B, local)',
    'codellama': 'CodeLlama\n(13B, local)',
    'mistral':   'Mistral\n(7B, local)',
    'phi3':      'Phi-3\n(3.8B, local)',
    'llama':     'Llama 3.1\n(70B, local)',
    'qwen':      'Qwen 2.5 Coder\n(32B, local)',
    'gpt4o':     'GPT-4o\n(API)',
    'claude':    'Claude 3.5 Sonnet\n(API)',
}

rows = []
for alias, label in models_display.items():
    key = f'sc_rag_3r_{alias}'
    if key in main:
        rows.append({
            'model': label,
            'functional': main[key]['functional_accuracy'],
            'security': main[key]['security_pass_rate'],
        })

if rows:
    df = pd.DataFrame(rows).sort_values('functional', ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df))
    ax.bar(x - 0.2, df['functional'], 0.4, label='Functional accuracy', color='#4472C4')
    ax.bar(x + 0.2, df['security'],   0.4, label='Security pass rate',  color='#ED7D31')
    ax.set_xticks(x)
    ax.set_xticklabels(df['model'], fontsize=9)
    ax.set_ylabel('Pass Rate (%)')
    ax.set_title('Model Comparison: sc+rag+3r condition')
    ax.legend()
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print('No sc_rag_3r results found in experiment_results.json')
    print('Available keys:', [k for k in main.keys() if 'rag' in k][:10])

## Model vs Condition Heatmap

In [ ]:
# Heatmap: rows=models, cols=conditions
conditions_map = {
    'one_shot': 'one-shot',
    'one_shot_rag': 'one-shot+rag',
    'sc_3r': 'sc+3r',
    'sc_rag_3r': 'sc+rag+3r',
    'sc_rag_5r': 'sc+rag+5r',
}
models_short = ['deepseek', 'codellama', 'mistral', 'phi3', 'llama', 'qwen', 'gpt4o', 'claude']

matrix = []
for model in models_short:
    row = []
    for cond_key in conditions_map:
        key = f'{cond_key}_{model}'
        val = main.get(key, {}).get('functional_accuracy', None)
        row.append(val if val is not None else 0)
    matrix.append(row)

if any(any(v > 0 for v in row) for row in matrix):
    fig, ax = plt.subplots(figsize=(9, 5))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=40, vmax=85)
    ax.set_xticks(range(len(conditions_map)))
    ax.set_xticklabels(list(conditions_map.values()), rotation=15, ha='right')
    ax.set_yticks(range(len(models_short)))
    ax.set_yticklabels(models_short)
    plt.colorbar(im, ax=ax, label='Functional Accuracy (%)')
    ax.set_title('Functional Accuracy Heatmap: Model × Condition')
    for i in range(len(models_short)):
        for j in range(len(conditions_map)):
            ax.text(j, i, f'{matrix[i][j]:.0f}', ha='center', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Matrix empty — check key format in experiment_results.json')